# Load libraries

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scikit-learn seaborn umap-learn scipy xgboost shap PyALE hyperopt

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps  # Ensure this is imported
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
import math
import json
import os
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
import shap
from PyALE import ale
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.stats import linregress, spearmanr
from itertools import product
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# Loading the data

In [ ]:
chamau_lag = pd.read_csv("../datasets/Chamau_2014-2024_clean_newlag.csv")
chamau_daily = pd.read_csv("../datasets/Chamau_Daily_2014-2024_newlag.csv")
oensingen_1_lag = pd.read_csv("../datasets/Oensingen_2018-19_clean_newlag.csv")
oensingen_1_daily = pd.read_csv("../datasets/Oensingen_Daily_2018-19_newlag.csv")
oensingen_2_lag = pd.read_csv("../datasets/Oensingen_2021-23_clean_newlag.csv")
oensingen_2_daily = pd.read_csv("../datasets/Oensingen_Daily_2021-23_newlag.csv")
aeschi_lag = pd.read_csv("../datasets/Aeschi_2019-20_clean_newlag.csv")
aeschi_daily = pd.read_csv("../datasets/Aeschi_Daily_2019-20_newlag.csv")
forel_lag = pd.read_csv("../datasets/Forel_2024-25_clean_newlag.csv")
forel_daily = pd.read_csv("../datasets/Forel_Daily_2024-25_newlag.csv")
tanikon_lag = pd.read_csv("../datasets/Tanikon_2023-25_clean_newlag.csv")
tanikon_daily = pd.read_csv("../datasets/Tanikon_Daily_2023-25_newlag.csv")

# Fine tuning functions

In [ ]:
def _evaluate_params_cv(X_trainval, y_trainval, params, offset, random_state, n_splits=5):
    """Evaluate parameters using time series cross-validation"""
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    r2_scores = []
    rho_scores = []
    
    for train_idx, val_idx in tscv.split(X_trainval):
        X_train_fold = X_trainval.iloc[train_idx]
        y_train_fold = y_trainval.iloc[train_idx]
        X_val_fold = X_trainval.iloc[val_idx]
        y_val_fold = y_trainval.iloc[val_idx]
        
        model = XGBRegressor(**params, random_state=random_state, n_jobs=-1)
        model.fit(X_train_fold, y_train_fold, verbose=False)
        y_pred = model.predict(X_val_fold)
        
        y_pred_lin = y_pred - offset
        y_val_lin = y_val_fold - offset
        
        r2 = r2_score(y_val_lin, y_pred_lin)
        rho, _ = spearmanr(y_val_lin, y_pred_lin)
        
        r2_scores.append(r2)
        rho_scores.append(rho)
    
    # Return mean scores across folds
    mean_r2 = np.mean(r2_scores)
    mean_rho = np.mean(rho_scores)
    
    return mean_r2, {'r2': mean_r2, 'spearman_rho': mean_rho}


def hyperparameter_search_timeseries(df, predictors, target, test_size=0.1, search_type='bayesian', n_iter=500, random_state=42, n_cv_splits=9):
    """
    Hyperparameter search using cross-validation on train+val data.
    
    Split strategy:
    - Train+Val (combined): 0 to (1 - test_size) - used for CV hyperparameter optimization
    - Test: (1 - test_size) to end - held out for final evaluation
    """

    X = df[predictors]
    y = df[target]
    mask = X.notna().all(axis=1) & y.notna()
    X, y = X[mask], y[mask]
    offset = abs(y.min()) + 1e-6
    y_shifted = y + offset
    
    # SPLIT: Train+Val (for CV) / Test
    split_idx_test = int(len(X) * (1 - test_size))
    
    X_trainval = X.iloc[:split_idx_test]
    y_trainval = y_shifted.iloc[:split_idx_test]
    
    X_test = X.iloc[split_idx_test:]
    y_test = y_shifted.iloc[split_idx_test:]
    
    print("Starting hyperparameter search with cross-validation...")
    print(f"Search type: Bayesian Optimization (TPE)")
    print(f"Iterations: {n_iter}")
    print(f"CV folds: {n_cv_splits}\n")
    print(f"Train+Val: {X_trainval.index[0]} → {X_trainval.index[-1]} ({len(X_trainval)} samples, {len(X_trainval)/len(X)*100:.1f}%)")
    print(f"Test:      {X_test.index[0]} → {X_test.index[-1]} ({len(X_test)} samples, {len(X_test)/len(X)*100:.1f}%)")
    print(f"\n⚠️  Using {n_cv_splits}-fold time series CV on Train+Val data")
    
    # Define hyperparameter search space
    space = {
        'n_estimators': hp.choice('n_estimators', [100, 200, 300, 500]),
        'max_depth': hp.choice('max_depth', [3, 4, 5, 6, 8]),
        'learning_rate': hp.choice('learning_rate', [0.01, 0.05, 0.1, 0.2]),
        'subsample': hp.choice('subsample', [0.6, 0.7, 0.8, 0.9]),
        'colsample_bytree': hp.choice('colsample_bytree', [0.6, 0.7, 0.8, 0.9]),
        'min_child_weight': hp.choice('min_child_weight', [1, 3, 5]),
        'gamma': hp.choice('gamma', [0, 0.1, 0.2, 0.5]),
    }
    
    all_results = []
    iteration_counter = [0]
    
    def objective(params):
        iteration_counter[0] += 1
        score, metrics = _evaluate_params_cv(X_trainval, y_trainval, params, offset, random_state, n_cv_splits)
        all_results.append({**params, **metrics})
        
        if iteration_counter[0] % 10 == 0:
            best_so_far = max([r['r2'] for r in all_results])
            print(f"Completed {iteration_counter[0]}/{n_iter} iterations. Best CV R²: {best_so_far:.4f}")
        
        return {'loss': -score, 'status': STATUS_OK}
    
    trials = Trials()
    best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=n_iter, trials=trials, rstate=np.random.default_rng(random_state), verbose=0)
    
    param_choices = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 4, 5, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.7, 0.8, 0.9],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
        'min_child_weight': [1, 3, 5],
        'gamma': [0, 0.1, 0.2, 0.5],
    }
    
    best_params = {k: param_choices[k][v] for k, v in best.items()}
    best_cv_score = max([r['r2'] for r in all_results])
    results_df = pd.DataFrame(all_results).sort_values('r2', ascending=False)
    
    print("\n" + "="*60)
    print("BAYESIAN OPTIMIZATION COMPLETE")
    print("="*60)
    print("\nBest parameters (selected via cross-validation):")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    print(f"\nBest CV R² (mean across {n_cv_splits} folds): {best_cv_score:.4f}")
    
    return best_params, results_df, X_trainval, y_trainval, X_test, y_test, offset

In [ ]:
def train_final_model(X_trainval, y_trainval, X_test, y_test, offset, predictors, best_params, random_state=42, plot=True):
    """
    Train final model on full train + val data, evaluate on test set.
    """
    
    test_dates = X_test.index
    
    print(f"\nTraining final model on {len(X_trainval)} samples (train+val combined)")
    
    model = XGBRegressor(**best_params, random_state=random_state, n_jobs=-1)
    model.fit(X_trainval, y_trainval, verbose=False)
    
    y_pred = model.predict(X_test)
    y_pred_lin = y_pred - offset
    y_test_lin = y_test - offset
    
    r2 = r2_score(y_test_lin, y_pred_lin)
    rho, _ = spearmanr(y_test_lin, y_pred_lin)
    
    importances = pd.Series(model.feature_importances_, index=predictors).sort_values(ascending=False)
    
    print("\n" + "="*60)
    print("FINAL MODEL PERFORMANCE (ON SAME TEST SET AS NORMAL MODEL)")
    print("="*60)
    print(f"R² (linear scale): {r2:.4f}")
    print(f"Spearman ρ:        {rho:.4f}")
    
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].scatter(y_test_lin, y_pred_lin, alpha=0.6, s=20)
        min_val, max_val = y_test_lin.min(), y_test_lin.max()
        axes[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
        axes[0].set_xlabel("Observed N₂O Flux")
        axes[0].set_ylabel("Predicted N₂O Flux")
        axes[0].set_title(f"Optimized XGBoost (Test R²={r2:.3f}, ρ={rho:.3f})")
        axes[0].grid(True, alpha=0.3)
        top_features = importances.head(15)
        axes[1].barh(range(len(top_features)), top_features.values)
        axes[1].set_yticks(range(len(top_features)))
        axes[1].set_yticklabels(top_features.index)
        axes[1].set_xlabel("Importance (Gain)")
        axes[1].set_title("Top 15 Features")
        axes[1].invert_yaxis()
        axes[2].plot(test_dates, y_test_lin.values, label="Observed", color="black", lw=1.5)
        axes[2].plot(test_dates, y_pred_lin, label="Predicted", color="royalblue", lw=1.5, alpha=0.8)
        axes[2].set_xlabel("Date")
        axes[2].set_ylabel("N₂O Flux")
        axes[2].set_title("Time Series Comparison (Test Set)")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
        plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    
    return {"model": model, "r2": r2, "spearman_rho": rho, "feature_importance": importances, "y_test": y_test_lin, "y_pred": y_pred_lin, "test_dates": test_dates, "best_params": best_params}

In [ ]:
def train_xgb_timeseries(df, predictors, target, test_size=0.1, n_estimators=300, random_state=42, plot=True):
    """
    Train XGBoost on time-series data using chronological split.
    
    Now includes a time series comparison plot showing observed vs predicted values over time.
    """
    
    # EXTRACT FEATURES/TARGET
    X = df[predictors]
    y = df[target]
    
    # HANDLE NaNs
    mask = X.notna().all(axis=1) & y.notna()
    X, y = X[mask], y[mask]
    
    # OFFSET FOR POSITIVITY
    offset = abs(y.min()) + 1e-6
    y_shifted = y + offset
    
    # CHRONOLOGICAL SPLIT
    split_idx = int(len(X) * (1 - test_size))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y_shifted.iloc[:split_idx], y_shifted.iloc[split_idx:]
    
    # Store test dates for time series plot
    test_dates = X_test.index
    
    time_train = (df.index.min(), df.index[:split_idx].max())
    time_test = (df.index[split_idx], df.index.max())
    
    print(f"Train period: {time_train[0]} → {time_train[1]}")
    print(f"Test  period: {time_test[0]} → {time_test[1]}")
    
    # ========== DATA LEAKAGE CHECK ==========
    try:
        # For hourly data with Timestamp index
        train_dates_set = set(X_train.index.date)
        test_dates_set = set(X_test.index.date)
    except AttributeError:
        # For daily data where index is integer, get dates from original df
        train_indices = X_train.index
        test_indices = X_test.index
        
        if 'Date' in df.columns:
            train_dates_set = set(pd.to_datetime(df.loc[train_indices, 'Date']).dt.date)
            test_dates_set = set(pd.to_datetime(df.loc[test_indices, 'Date']).dt.date)
        else:
            train_dates_set = set()
            test_dates_set = set()
            print("  ⚠️  Cannot check for data leakage (no Date column found)")
    
    overlap_dates = train_dates_set & test_dates_set
    if len(overlap_dates) > 0:
        print(f"  ⚠️  DATA LEAKAGE WARNING: {len(overlap_dates)} dates appear in BOTH train and test!")
    else:
        print(f"  ✅ No data leakage detected")
    # =========================================
    
    # TRAIN MODEL
    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=random_state,
        n_jobs=-1
    )
    model.fit(X_train, y_train, verbose=False)
    
    # PREDICTIONS
    y_pred = model.predict(X_test)
    
    # REVERT OFFSET
    y_pred_lin = y_pred - offset
    y_test_lin = y_test - offset
    
    # EVALUATION
    r2 = r2_score(y_test_lin, y_pred_lin)
    rho, _ = spearmanr(y_test_lin, y_pred_lin)
    
    # FEATURE IMPORTANCE
    importances = pd.Series(model.feature_importances_, index=predictors).sort_values(ascending=False)
    
    print("\nModel evaluation:")
    print(f"  R² (linear scale): {r2:.3f}")
    print(f"  Spearman ρ:        {rho:.3f}")
    
    # PLOT
    if plot:
        # Create 3-subplot figure: scatter, feature importance, and TIME SERIES
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # 1. Scatter plot
        axes[0].scatter(y_test_lin, y_pred_lin, alpha=0.6, s=20)
        min_val, max_val = y_test_lin.min(), y_test_lin.max()
        axes[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
        axes[0].set_xlabel("Observed N₂O Flux")
        axes[0].set_ylabel("Predicted N₂O Flux")
        axes[0].set_title(f"XGBoost (R²={r2:.3f}, ρ={rho:.3f})")
        axes[0].grid(True, alpha=0.3)
        
        # 2. Feature importance
        top_features = importances.head(15)
        axes[1].barh(range(len(top_features)), top_features.values)
        axes[1].set_yticks(range(len(top_features)))
        axes[1].set_yticklabels(top_features.index)
        axes[1].set_xlabel("Importance (Gain)")
        axes[1].set_title("Top 15 Features")
        axes[1].invert_yaxis()
        
        # 3. TIME SERIES COMPARISON (NEW!)
        axes[2].plot(test_dates, y_test_lin.values, label="Observed", color="black", lw=1.5)
        axes[2].plot(test_dates, y_pred_lin, label="Predicted", color="royalblue", lw=1.5, alpha=0.8)
        axes[2].set_xlabel("Date")
        axes[2].set_ylabel("N₂O Flux")
        axes[2].set_title("Time Series Comparison")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
        
        # Rotate x-axis labels for better readability
        plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
    
    return {
        "model": model,
        "r2": r2,
        "spearman_rho": rho,
        "feature_importance": importances,
        "y_test": y_test_lin,
        "y_pred": y_pred_lin,
        "test_dates": test_dates,  # Also return test dates
    }

# Experiment 1: Chamau dataset

In [ ]:
# FIX TIMESTAMP
chamau_lag['Timestamp'] = pd.to_datetime(chamau_lag['Timestamp'])
chamau_lag = chamau_lag.set_index('Timestamp').sort_index()

chamau_daily['Date'] = pd.to_datetime(chamau_daily['Date'])
chamau_daily = chamau_daily.set_index('Date').sort_index()

In [ ]:
# Define base and lag predictors
base_predictors = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm",
    "SoilTemp_4cm", "SoilTemp_15cm",
    "NEE", "GPP", "RECO",
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "Grazing", "SoilCultivation", "Fertilizer_N_kg_ha"
]

lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm",
    "SoilTemp_4cm", "SoilTemp_15cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilTemp_4cm_lag1d_daily", "SoilTemp_4cm_lag3d_daily", "SoilTemp_4cm_lag5d_daily", "SoilTemp_4cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",

    # --- Management variables ---
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "Grazing", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_Grazing", "DaysSince_SoilCultivation", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilTemp_4cm", "SoilTemp_15cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilTemp_4cm_lag1d_daily", "SoilTemp_4cm_lag3d_daily", "SoilTemp_4cm_lag5d_daily", "SoilTemp_4cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilTemp_4cm_roll3d_mean", "SoilTemp_4cm_roll5d_mean", "SoilTemp_4cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilTemp_4cm_roll3d_sum", "SoilTemp_4cm_roll5d_sum", "SoilTemp_4cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # --- Management variables ---
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "Grazing", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_Grazing", "DaysSince_SoilCultivation", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

In [ ]:
configurations = [
    ("Hourly", "Base", chamau_lag, base_predictors),
    ("Hourly", "Lag", chamau_lag, lag_predictors),
    ("Hourly", "Augmented", chamau_lag, augmented_predictors),
    ("Daily", "Base", chamau_daily, base_predictors),
    ("Daily", "Lag", chamau_daily, lag_predictors),
    ("Daily", "Augmented", chamau_daily, augmented_predictors),
]

all_optimized_results = []
all_best_params = {}

for temporal, pred_type, df, predictors in configurations:
    print("\n" + "="*80)
    print(f"OPTIMIZING: {temporal} | {pred_type}")
    print("="*80)
    
    # CORRECTED: Only 7 return values now
    best_params, search_results, X_trainval, y_trainval, X_test, y_test, offset = hyperparameter_search_timeseries(
        df=df, 
        predictors=predictors, 
        target="N2O_Flux_ln", 
        test_size=0.1,
        search_type='bayesian',
        n_iter=500,
        random_state=42,
        n_cv_splits=9
    )
    
    all_best_params[f"{temporal}_{pred_type}"] = best_params
    
    # CORRECTED: train_final_model also needs updating
    final_results = train_final_model(
        X_trainval=X_trainval,  # Changed from X_train
        y_trainval=y_trainval,  # Changed from y_train
        X_test=X_test,
        y_test=y_test,
        offset=offset,
        predictors=predictors,
        best_params=best_params,
        random_state=42,
        plot=True
    )
    
    all_optimized_results.append({
        'Temporal': temporal, 
        'Predictors': pred_type, 
        'R2_optimized': final_results['r2'], 
        'Spearman_optimized': final_results['spearman_rho'], 
        'Best_Params': best_params
    })
    
    print(f"\n✓ Completed {temporal} | {pred_type}")
    print(f"  Test R²: {final_results['r2']:.4f}")
    print(f"  Test ρ: {final_results['spearman_rho']:.4f}")

In [ ]:
# Re-run original models for comparison
original_results = []
all_original_params = {}  # ADD THIS: Store original parameters

for temporal, pred_type, df, predictors in configurations:
    print(f"\nRunning original {temporal} | {pred_type}...")
    
    orig = train_xgb_timeseries(
        df=df,
        predictors=predictors,
        target="N2O_Flux_ln",
        test_size=0.1,
        n_estimators=300,
        random_state=42,
        plot=False
    )
    

    # Store original parameters
    config_name = f"{temporal}_{pred_type}"
    all_original_params[config_name] = {
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.1,  # ← FIXED
        'subsample': 0.8,      # ← FIXED
        'colsample_bytree': 0.8,  # ← FIXED
        'min_child_weight': 1,
        'gamma': 0,
        'random_state': 42
    }
    
    original_results.append({
        'Temporal': temporal,
        'Predictors': pred_type,
        'R2_original': orig['r2'],
        'Spearman_original': orig['spearman_rho']
    })

# Combine results
comparison_df = pd.merge(
    pd.DataFrame(original_results),
    pd.DataFrame(all_optimized_results),
    on=['Temporal', 'Predictors']
)

# Calculate improvements
comparison_df['R2_improvement'] = comparison_df['R2_optimized'] - comparison_df['R2_original']
comparison_df['Spearman_improvement'] = comparison_df['Spearman_optimized'] - comparison_df['Spearman_original']

# ADD THIS: Flag which model is better
comparison_df['Better_Model'] = comparison_df.apply(
    lambda row: 'Optimized' if row['R2_optimized'] > row['R2_original'] else 'Original', 
    axis=1
)

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ORIGINAL VS OPTIMIZED")
print("="*80)
print(comparison_df[['Temporal', 'Predictors', 'R2_original', 'R2_optimized', 
                     'R2_improvement', 'Spearman_original', 'Spearman_optimized', 
                     'Spearman_improvement', 'Better_Model']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x - width/2, comparison_df['R2_original'], width, label='Original', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['R2_optimized'], width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Configuration')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Score: Original vs Optimized')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(x - width/2, comparison_df['Spearman_original'], width, label='Original', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['Spearman_optimized'], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Spearman ρ')
axes[1].set_title('Spearman ρ: Original vs Optimized')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Define dataset name
dataset_name = "Chamau"

# Create dataset subfolder inside existing xgboost_params folder
save_dir = os.path.join('xgboost_params', dataset_name)
os.makedirs(save_dir, exist_ok=True)

# Save comparison dataframe
comparison_df.to_csv(os.path.join(save_dir, 'optimization_comparison.csv'), index=False)

# Save both original and optimized parameters as JSON
with open(os.path.join(save_dir, 'original_hyperparameters.json'), 'w') as f:
    json.dump(all_original_params, f, indent=4)

with open(os.path.join(save_dir, 'optimized_hyperparameters.json'), 'w') as f:
    json.dump(all_best_params, f, indent=4)

# ADD THIS: Save best performing parameters (choose better of original vs optimized)
all_best_performing_params = {}
for _, row in comparison_df.iterrows():
    config_name = f"{row['Temporal']}_{row['Predictors']}"
    if row['Better_Model'] == 'Optimized':
        all_best_performing_params[config_name] = all_best_params[config_name]
    else:
        all_best_performing_params[config_name] = all_original_params[config_name]

with open(os.path.join(save_dir, 'best_performing_hyperparameters.json'), 'w') as f:
    json.dump(all_best_performing_params, f, indent=4)

# Save each configuration's params as separate JSON files
for config_name in all_best_params.keys():
    # Original params
    with open(os.path.join(save_dir, f'{config_name}_original_params.json'), 'w') as f:
        json.dump(all_original_params[config_name], f, indent=4)
    
    # Optimized params
    with open(os.path.join(save_dir, f'{config_name}_optimized_params.json'), 'w') as f:
        json.dump(all_best_params[config_name], f, indent=4)
    
    # Best performing params
    with open(os.path.join(save_dir, f'{config_name}_best_params.json'), 'w') as f:
        json.dump(all_best_performing_params[config_name], f, indent=4)

# Save detailed comparison with both parameter sets
comparison_with_params = comparison_df.copy()
comparison_with_params['Original_Params'] = comparison_with_params.apply(
    lambda row: all_original_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Optimized_Params'] = comparison_with_params.apply(
    lambda row: all_best_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Best_Performing_Params'] = comparison_with_params.apply(
    lambda row: all_best_performing_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params.to_csv(os.path.join(save_dir, 'detailed_comparison_with_params.csv'), index=False)

print(f"\nResults saved to {save_dir}!")
print("  - optimization_comparison.csv")
print("  - original_hyperparameters.json")
print("  - optimized_hyperparameters.json")
print("  - best_performing_hyperparameters.json (automatically selects better model)")
print("  - detailed_comparison_with_params.csv")
print("  - Individual config param files (18 files: 6 original + 6 optimized + 6 best)")
print(f"\nTotal files saved: {len(os.listdir(save_dir))}")
print(f"Full path: {os.path.abspath(save_dir)}")

# Print summary of which models performed better
print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)
for _, row in comparison_df.iterrows():
    winner = "✓ OPTIMIZED" if row['Better_Model'] == 'Optimized' else "✓ ORIGINAL"
    print(f"{row['Temporal']:15} | {row['Predictors']:20} | {winner:12} | ΔR²: {row['R2_improvement']:+.4f}")

# Experiment 2: Oensingen 2021-23 dataset

In [ ]:
# FIX TIMESTAMP
oensingen_2_lag['Timestamp'] = pd.to_datetime(oensingen_2_lag['Timestamp'])
oensingen_2_lag = oensingen_2_lag.set_index('Timestamp').sort_index()

oensingen_2_daily['Date'] = pd.to_datetime(oensingen_2_daily['Date'])
oensingen_2_daily = oensingen_2_daily.set_index('Date').sort_index()

In [ ]:
# Define base and lag predictors
base_predictors = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation", "Fertilizer_N_kg_ha"
]

hourly_lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilWater_30cm_lag1d_daily", "SoilWater_30cm_lag3d_daily", "SoilWater_30cm_lag5d_daily", "SoilWater_30cm_lag7d_daily",
    "SoilTemp_5cm_lag1d_daily", "SoilTemp_5cm_lag3d_daily", "SoilTemp_5cm_lag5d_daily", "SoilTemp_5cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "SoilTemp_30cm_lag1d_daily", "SoilTemp_30cm_lag3d_daily", "SoilTemp_30cm_lag5d_daily", "SoilTemp_30cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",

    # --- Management variables ---
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

hourly_augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilWater_30cm_lag1d_daily", "SoilWater_30cm_lag3d_daily", "SoilWater_30cm_lag5d_daily", "SoilWater_30cm_lag7d_daily",
    "SoilTemp_5cm_lag1d_daily", "SoilTemp_5cm_lag3d_daily", "SoilTemp_5cm_lag5d_daily", "SoilTemp_5cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "SoilTemp_30cm_lag1d_daily", "SoilTemp_30cm_lag3d_daily", "SoilTemp_30cm_lag5d_daily", "SoilTemp_30cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilWater_30cm_roll3d_mean", "SoilWater_30cm_roll5d_mean", "SoilWater_30cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "SoilTemp_30cm_roll3d_mean", "SoilTemp_30cm_roll5d_mean", "SoilTemp_30cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilWater_30cm_roll3d_sum", "SoilWater_30cm_roll5d_sum", "SoilWater_30cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "SoilTemp_30cm_roll3d_sum", "SoilTemp_30cm_roll5d_sum", "SoilTemp_30cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # Management events and days since
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

daily_lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",

    # --- Management variables ---
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

daily_augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilWater_30cm_roll3d_mean", "SoilWater_30cm_roll5d_mean", "SoilWater_30cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "SoilTemp_30cm_roll3d_mean", "SoilTemp_30cm_roll5d_mean", "SoilTemp_30cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilWater_30cm_roll3d_sum", "SoilWater_30cm_roll5d_sum", "SoilWater_30cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "SoilTemp_30cm_roll3d_sum", "SoilTemp_30cm_roll5d_sum", "SoilTemp_30cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # Management events and days since
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

In [ ]:
configurations = [
    ("Hourly", "Base", oensingen_2_lag, base_predictors),
    ("Hourly", "Lag", oensingen_2_lag, hourly_lag_predictors),
    ("Hourly", "Augmented", oensingen_2_lag, hourly_augmented_predictors),
    ("Daily", "Base", oensingen_2_daily, base_predictors),
    ("Daily", "Lag", oensingen_2_daily, daily_lag_predictors),
    ("Daily", "Augmented", oensingen_2_daily, daily_augmented_predictors),
]

all_optimized_results = []
all_best_params = {}

for temporal, pred_type, df, predictors in configurations:
    print("\n" + "="*80)
    print(f"OPTIMIZING: {temporal} | {pred_type}")
    print("="*80)
    
    # CORRECTED: Only 7 return values now
    best_params, search_results, X_trainval, y_trainval, X_test, y_test, offset = hyperparameter_search_timeseries(
        df=df, 
        predictors=predictors, 
        target="N2O_Flux_ln", 
        test_size=0.1,
        search_type='bayesian',
        n_iter=500,
        random_state=42,
        n_cv_splits=9
    )
    
    all_best_params[f"{temporal}_{pred_type}"] = best_params
    
    # CORRECTED: train_final_model also needs updating
    final_results = train_final_model(
        X_trainval=X_trainval,  # Changed from X_train
        y_trainval=y_trainval,  # Changed from y_train
        X_test=X_test,
        y_test=y_test,
        offset=offset,
        predictors=predictors,
        best_params=best_params,
        random_state=42,
        plot=True
    )
    
    all_optimized_results.append({
        'Temporal': temporal, 
        'Predictors': pred_type, 
        'R2_optimized': final_results['r2'], 
        'Spearman_optimized': final_results['spearman_rho'], 
        'Best_Params': best_params
    })
    
    print(f"\n✓ Completed {temporal} | {pred_type}")
    print(f"  Test R²: {final_results['r2']:.4f}")
    print(f"  Test ρ: {final_results['spearman_rho']:.4f}")

In [ ]:
# Re-run original models for comparison
original_results = []
all_original_params = {}  # ADD THIS: Store original parameters

for temporal, pred_type, df, predictors in configurations:
    print(f"\nRunning original {temporal} | {pred_type}...")
    
    orig = train_xgb_timeseries(
        df=df,
        predictors=predictors,
        target="N2O_Flux_ln",
        test_size=0.1,
        n_estimators=300,
        random_state=42,
        plot=False
    )
    
    # Store original parameters
    config_name = f"{temporal}_{pred_type}"
    all_original_params[config_name] = {
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.1,  # ← FIXED
        'subsample': 0.8,      # ← FIXED
        'colsample_bytree': 0.8,  # ← FIXED
        'min_child_weight': 1,
        'gamma': 0,
        'random_state': 42
    }
    
    original_results.append({
        'Temporal': temporal,
        'Predictors': pred_type,
        'R2_original': orig['r2'],
        'Spearman_original': orig['spearman_rho']
    })

# Combine results
comparison_df = pd.merge(
    pd.DataFrame(original_results),
    pd.DataFrame(all_optimized_results),
    on=['Temporal', 'Predictors']
)

# Calculate improvements
comparison_df['R2_improvement'] = comparison_df['R2_optimized'] - comparison_df['R2_original']
comparison_df['Spearman_improvement'] = comparison_df['Spearman_optimized'] - comparison_df['Spearman_original']

# ADD THIS: Flag which model is better
comparison_df['Better_Model'] = comparison_df.apply(
    lambda row: 'Optimized' if row['R2_optimized'] > row['R2_original'] else 'Original', 
    axis=1
)

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ORIGINAL VS OPTIMIZED")
print("="*80)
print(comparison_df[['Temporal', 'Predictors', 'R2_original', 'R2_optimized', 
                     'R2_improvement', 'Spearman_original', 'Spearman_optimized', 
                     'Spearman_improvement', 'Better_Model']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x - width/2, comparison_df['R2_original'], width, label='Original', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['R2_optimized'], width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Configuration')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Score: Original vs Optimized')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(x - width/2, comparison_df['Spearman_original'], width, label='Original', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['Spearman_optimized'], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Spearman ρ')
axes[1].set_title('Spearman ρ: Original vs Optimized')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Define dataset name
dataset_name = "Oensingen_2021-23"

# Create dataset subfolder inside existing xgboost_params folder
save_dir = os.path.join('xgboost_params', dataset_name)
os.makedirs(save_dir, exist_ok=True)

# Save comparison dataframe
comparison_df.to_csv(os.path.join(save_dir, 'optimization_comparison.csv'), index=False)

# Save both original and optimized parameters as JSON
with open(os.path.join(save_dir, 'original_hyperparameters.json'), 'w') as f:
    json.dump(all_original_params, f, indent=4)

with open(os.path.join(save_dir, 'optimized_hyperparameters.json'), 'w') as f:
    json.dump(all_best_params, f, indent=4)

# ADD THIS: Save best performing parameters (choose better of original vs optimized)
all_best_performing_params = {}
for _, row in comparison_df.iterrows():
    config_name = f"{row['Temporal']}_{row['Predictors']}"
    if row['Better_Model'] == 'Optimized':
        all_best_performing_params[config_name] = all_best_params[config_name]
    else:
        all_best_performing_params[config_name] = all_original_params[config_name]

with open(os.path.join(save_dir, 'best_performing_hyperparameters.json'), 'w') as f:
    json.dump(all_best_performing_params, f, indent=4)

# Save each configuration's params as separate JSON files
for config_name in all_best_params.keys():
    # Original params
    with open(os.path.join(save_dir, f'{config_name}_original_params.json'), 'w') as f:
        json.dump(all_original_params[config_name], f, indent=4)
    
    # Optimized params
    with open(os.path.join(save_dir, f'{config_name}_optimized_params.json'), 'w') as f:
        json.dump(all_best_params[config_name], f, indent=4)
    
    # Best performing params
    with open(os.path.join(save_dir, f'{config_name}_best_params.json'), 'w') as f:
        json.dump(all_best_performing_params[config_name], f, indent=4)

# Save detailed comparison with both parameter sets
comparison_with_params = comparison_df.copy()
comparison_with_params['Original_Params'] = comparison_with_params.apply(
    lambda row: all_original_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Optimized_Params'] = comparison_with_params.apply(
    lambda row: all_best_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Best_Performing_Params'] = comparison_with_params.apply(
    lambda row: all_best_performing_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params.to_csv(os.path.join(save_dir, 'detailed_comparison_with_params.csv'), index=False)

print(f"\nResults saved to {save_dir}!")
print("  - optimization_comparison.csv")
print("  - original_hyperparameters.json")
print("  - optimized_hyperparameters.json")
print("  - best_performing_hyperparameters.json (automatically selects better model)")
print("  - detailed_comparison_with_params.csv")
print("  - Individual config param files (18 files: 6 original + 6 optimized + 6 best)")
print(f"\nTotal files saved: {len(os.listdir(save_dir))}")
print(f"Full path: {os.path.abspath(save_dir)}")

# Print summary of which models performed better
print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)
for _, row in comparison_df.iterrows():
    winner = "✓ OPTIMIZED" if row['Better_Model'] == 'Optimized' else "✓ ORIGINAL"
    print(f"{row['Temporal']:15} | {row['Predictors']:20} | {winner:12} | ΔR²: {row['R2_improvement']:+.4f}")

# Experiment 3: Aeschi dataset

In [ ]:
# FIX TIMESTAMP
aeschi_lag['Timestamp'] = pd.to_datetime(aeschi_lag['Timestamp'])
aeschi_lag = aeschi_lag.set_index('Timestamp').sort_index()

aeschi_daily['Date'] = pd.to_datetime(aeschi_daily['Date'])
aeschi_daily = aeschi_daily.set_index('Date').sort_index()

In [ ]:
base_predictors = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    "Mowing",
]

lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",

    # --- Management variables ---
    "Mowing", "DaysSince_Mowing"
]

augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # --- Management variables ---
    "Mowing", "DaysSince_Mowing"
]

In [ ]:
configurations = [
    ("Hourly", "Base", aeschi_lag, base_predictors),
    ("Hourly", "Lag", aeschi_lag, lag_predictors),
    ("Hourly", "Augmented", aeschi_lag, augmented_predictors),
    ("Daily", "Base", aeschi_daily, base_predictors),
    ("Daily", "Lag", aeschi_daily, lag_predictors),
    ("Daily", "Augmented", aeschi_daily, augmented_predictors),
]

all_optimized_results = []
all_best_params = {}

for temporal, pred_type, df, predictors in configurations:
    print("\n" + "="*80)
    print(f"OPTIMIZING: {temporal} | {pred_type}")
    print("="*80)
    
    # CORRECTED: Only 7 return values now
    best_params, search_results, X_trainval, y_trainval, X_test, y_test, offset = hyperparameter_search_timeseries(
        df=df, 
        predictors=predictors, 
        target="N2O_Flux_ln", 
        test_size=0.1,
        search_type='bayesian',
        n_iter=500,
        random_state=42,
        n_cv_splits=9
    )
    
    all_best_params[f"{temporal}_{pred_type}"] = best_params
    
    # CORRECTED: train_final_model also needs updating
    final_results = train_final_model(
        X_trainval=X_trainval,  # Changed from X_train
        y_trainval=y_trainval,  # Changed from y_train
        X_test=X_test,
        y_test=y_test,
        offset=offset,
        predictors=predictors,
        best_params=best_params,
        random_state=42,
        plot=True
    )
    
    all_optimized_results.append({
        'Temporal': temporal, 
        'Predictors': pred_type, 
        'R2_optimized': final_results['r2'], 
        'Spearman_optimized': final_results['spearman_rho'], 
        'Best_Params': best_params
    })
    
    print(f"\n✓ Completed {temporal} | {pred_type}")
    print(f"  Test R²: {final_results['r2']:.4f}")
    print(f"  Test ρ: {final_results['spearman_rho']:.4f}")

In [ ]:
# Re-run original models for comparison
original_results = []
all_original_params = {}  # ADD THIS: Store original parameters

for temporal, pred_type, df, predictors in configurations:
    print(f"\nRunning original {temporal} | {pred_type}...")
    
    orig = train_xgb_timeseries(
        df=df,
        predictors=predictors,
        target="N2O_Flux_ln",
        test_size=0.1,
        n_estimators=300,
        random_state=42,
        plot=False
    )
    
    # Store original parameters
    config_name = f"{temporal}_{pred_type}"
    all_original_params[config_name] = {
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.1,  # ← FIXED
        'subsample': 0.8,      # ← FIXED
        'colsample_bytree': 0.8,  # ← FIXED
        'min_child_weight': 1,
        'gamma': 0,
        'random_state': 42
    }
    
    original_results.append({
        'Temporal': temporal,
        'Predictors': pred_type,
        'R2_original': orig['r2'],
        'Spearman_original': orig['spearman_rho']
    })

# Combine results
comparison_df = pd.merge(
    pd.DataFrame(original_results),
    pd.DataFrame(all_optimized_results),
    on=['Temporal', 'Predictors']
)

# Calculate improvements
comparison_df['R2_improvement'] = comparison_df['R2_optimized'] - comparison_df['R2_original']
comparison_df['Spearman_improvement'] = comparison_df['Spearman_optimized'] - comparison_df['Spearman_original']

# ADD THIS: Flag which model is better
comparison_df['Better_Model'] = comparison_df.apply(
    lambda row: 'Optimized' if row['R2_optimized'] > row['R2_original'] else 'Original', 
    axis=1
)

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ORIGINAL VS OPTIMIZED")
print("="*80)
print(comparison_df[['Temporal', 'Predictors', 'R2_original', 'R2_optimized', 
                     'R2_improvement', 'Spearman_original', 'Spearman_optimized', 
                     'Spearman_improvement', 'Better_Model']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x - width/2, comparison_df['R2_original'], width, label='Original', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['R2_optimized'], width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Configuration')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Score: Original vs Optimized')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].bar(x - width/2, comparison_df['Spearman_original'], width, label='Original', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['Spearman_optimized'], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Spearman ρ')
axes[1].set_title('Spearman ρ: Original vs Optimized')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Define dataset name
dataset_name = "Aeschi"

# Create dataset subfolder inside existing xgboost_params folder
save_dir = os.path.join('xgboost_params', dataset_name)
os.makedirs(save_dir, exist_ok=True)

# Save comparison dataframe
comparison_df.to_csv(os.path.join(save_dir, 'optimization_comparison.csv'), index=False)

# Save both original and optimized parameters as JSON
with open(os.path.join(save_dir, 'original_hyperparameters.json'), 'w') as f:
    json.dump(all_original_params, f, indent=4)

with open(os.path.join(save_dir, 'optimized_hyperparameters.json'), 'w') as f:
    json.dump(all_best_params, f, indent=4)

# ADD THIS: Save best performing parameters (choose better of original vs optimized)
all_best_performing_params = {}
for _, row in comparison_df.iterrows():
    config_name = f"{row['Temporal']}_{row['Predictors']}"
    if row['Better_Model'] == 'Optimized':
        all_best_performing_params[config_name] = all_best_params[config_name]
    else:
        all_best_performing_params[config_name] = all_original_params[config_name]

with open(os.path.join(save_dir, 'best_performing_hyperparameters.json'), 'w') as f:
    json.dump(all_best_performing_params, f, indent=4)

# Save each configuration's params as separate JSON files
for config_name in all_best_params.keys():
    # Original params
    with open(os.path.join(save_dir, f'{config_name}_original_params.json'), 'w') as f:
        json.dump(all_original_params[config_name], f, indent=4)
    
    # Optimized params
    with open(os.path.join(save_dir, f'{config_name}_optimized_params.json'), 'w') as f:
        json.dump(all_best_params[config_name], f, indent=4)
    
    # Best performing params
    with open(os.path.join(save_dir, f'{config_name}_best_params.json'), 'w') as f:
        json.dump(all_best_performing_params[config_name], f, indent=4)

# Save detailed comparison with both parameter sets
comparison_with_params = comparison_df.copy()
comparison_with_params['Original_Params'] = comparison_with_params.apply(
    lambda row: all_original_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Optimized_Params'] = comparison_with_params.apply(
    lambda row: all_best_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Best_Performing_Params'] = comparison_with_params.apply(
    lambda row: all_best_performing_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params.to_csv(os.path.join(save_dir, 'detailed_comparison_with_params.csv'), index=False)

print(f"\nResults saved to {save_dir}!")
print("  - optimization_comparison.csv")
print("  - original_hyperparameters.json")
print("  - optimized_hyperparameters.json")
print("  - best_performing_hyperparameters.json (automatically selects better model)")
print("  - detailed_comparison_with_params.csv")
print("  - Individual config param files (18 files: 6 original + 6 optimized + 6 best)")
print(f"\nTotal files saved: {len(os.listdir(save_dir))}")
print(f"Full path: {os.path.abspath(save_dir)}")

# Print summary of which models performed better
print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)
for _, row in comparison_df.iterrows():
    winner = "✓ OPTIMIZED" if row['Better_Model'] == 'Optimized' else "✓ ORIGINAL"
    print(f"{row['Temporal']:15} | {row['Predictors']:20} | {winner:12} | ΔR²: {row['R2_improvement']:+.4f}")

# Experiment 4: Oensingen 2018-19 dataset

In [ ]:
# FIX TIMESTAMP
oensingen_1_lag['Timestamp'] = pd.to_datetime(oensingen_1_lag['Timestamp'])
oensingen_1_lag = oensingen_1_lag.set_index('Timestamp').sort_index()

oensingen_1_daily['Date'] = pd.to_datetime(oensingen_1_daily['Date'])
oensingen_1_daily = oensingen_1_daily.set_index('Date').sort_index()

In [ ]:
# Define base and lag predictors
base_predictors = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    "Mowing", "SoilCultivation"
]

hourly_lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilWater_30cm_lag1d_daily", "SoilWater_30cm_lag3d_daily", "SoilWater_30cm_lag5d_daily", "SoilWater_30cm_lag7d_daily",
    "SoilTemp_5cm_lag1d_daily", "SoilTemp_5cm_lag3d_daily", "SoilTemp_5cm_lag5d_daily", "SoilTemp_5cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "SoilTemp_30cm_lag1d_daily", "SoilTemp_30cm_lag3d_daily", "SoilTemp_30cm_lag5d_daily", "SoilTemp_30cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",

    # --- Management variables ---
    "Mowing", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_SoilCultivation"
]

hourly_augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilWater_30cm_lag1d_daily", "SoilWater_30cm_lag3d_daily", "SoilWater_30cm_lag5d_daily", "SoilWater_30cm_lag7d_daily",
    "SoilTemp_5cm_lag1d_daily", "SoilTemp_5cm_lag3d_daily", "SoilTemp_5cm_lag5d_daily", "SoilTemp_5cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "SoilTemp_30cm_lag1d_daily", "SoilTemp_30cm_lag3d_daily", "SoilTemp_30cm_lag5d_daily", "SoilTemp_30cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilWater_30cm_roll3d_mean", "SoilWater_30cm_roll5d_mean", "SoilWater_30cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "SoilTemp_30cm_roll3d_mean", "SoilTemp_30cm_roll5d_mean", "SoilTemp_30cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilWater_30cm_roll3d_sum", "SoilWater_30cm_roll5d_sum", "SoilWater_30cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "SoilTemp_30cm_roll3d_sum", "SoilTemp_30cm_roll5d_sum", "SoilTemp_30cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # --- Management variables ---
    "Mowing", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_SoilCultivation"
]

daily_lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",

    # --- Management variables ---
    "Mowing", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_SoilCultivation"
]

daily_augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilWater_30cm_roll3d_mean", "SoilWater_30cm_roll5d_mean", "SoilWater_30cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "SoilTemp_30cm_roll3d_mean", "SoilTemp_30cm_roll5d_mean", "SoilTemp_30cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilWater_30cm_roll3d_sum", "SoilWater_30cm_roll5d_sum", "SoilWater_30cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "SoilTemp_30cm_roll3d_sum", "SoilTemp_30cm_roll5d_sum", "SoilTemp_30cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # --- Management variables ---
    "Mowing", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_SoilCultivation"
]

In [ ]:
configurations = [
    ("Hourly", "Base", oensingen_1_lag, base_predictors),
    ("Hourly", "Lag", oensingen_1_lag, hourly_lag_predictors),
    ("Hourly", "Augmented", oensingen_1_lag, hourly_augmented_predictors),
    ("Daily", "Base", oensingen_1_daily, base_predictors),
    ("Daily", "Lag", oensingen_1_daily, daily_lag_predictors),
    ("Daily", "Augmented", oensingen_1_daily, daily_augmented_predictors),
]

all_optimized_results = []
all_best_params = {}

for temporal, pred_type, df, predictors in configurations:
    print("\n" + "="*80)
    print(f"OPTIMIZING: {temporal} | {pred_type}")
    print("="*80)
    
    # CORRECTED: Only 7 return values now
    best_params, search_results, X_trainval, y_trainval, X_test, y_test, offset = hyperparameter_search_timeseries(
        df=df, 
        predictors=predictors, 
        target="N2O_Flux_ln", 
        test_size=0.1,
        search_type='bayesian',
        n_iter=500,
        random_state=42,
        n_cv_splits=9
    )
    
    all_best_params[f"{temporal}_{pred_type}"] = best_params
    
    # CORRECTED: train_final_model also needs updating
    final_results = train_final_model(
        X_trainval=X_trainval,  # Changed from X_train
        y_trainval=y_trainval,  # Changed from y_train
        X_test=X_test,
        y_test=y_test,
        offset=offset,
        predictors=predictors,
        best_params=best_params,
        random_state=42,
        plot=True
    )
    
    all_optimized_results.append({
        'Temporal': temporal, 
        'Predictors': pred_type, 
        'R2_optimized': final_results['r2'], 
        'Spearman_optimized': final_results['spearman_rho'], 
        'Best_Params': best_params
    })
    
    print(f"\n✓ Completed {temporal} | {pred_type}")
    print(f"  Test R²: {final_results['r2']:.4f}")
    print(f"  Test ρ: {final_results['spearman_rho']:.4f}")

In [ ]:
# Re-run original models for comparison
original_results = []
all_original_params = {}  # ADD THIS: Store original parameters

for temporal, pred_type, df, predictors in configurations:
    print(f"\nRunning original {temporal} | {pred_type}...")
    
    orig = train_xgb_timeseries(
        df=df,
        predictors=predictors,
        target="N2O_Flux_ln",
        test_size=0.1,
        n_estimators=300,
        random_state=42,
        plot=False
    )
    
    # Store original parameters
    config_name = f"{temporal}_{pred_type}"
    all_original_params[config_name] = {
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.1,  # ← FIXED
        'subsample': 0.8,      # ← FIXED
        'colsample_bytree': 0.8,  # ← FIXED
        'min_child_weight': 1,
        'gamma': 0,
        'random_state': 42
    }
    
    original_results.append({
        'Temporal': temporal,
        'Predictors': pred_type,
        'R2_original': orig['r2'],
        'Spearman_original': orig['spearman_rho']
    })

# Combine results
comparison_df = pd.merge(
    pd.DataFrame(original_results),
    pd.DataFrame(all_optimized_results),
    on=['Temporal', 'Predictors']
)

# Calculate improvements
comparison_df['R2_improvement'] = comparison_df['R2_optimized'] - comparison_df['R2_original']
comparison_df['Spearman_improvement'] = comparison_df['Spearman_optimized'] - comparison_df['Spearman_original']

# ADD THIS: Flag which model is better
comparison_df['Better_Model'] = comparison_df.apply(
    lambda row: 'Optimized' if row['R2_optimized'] > row['R2_original'] else 'Original', 
    axis=1
)

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ORIGINAL VS OPTIMIZED")
print("="*80)
print(comparison_df[['Temporal', 'Predictors', 'R2_original', 'R2_optimized', 
                     'R2_improvement', 'Spearman_original', 'Spearman_optimized', 
                     'Spearman_improvement', 'Better_Model']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x - width/2, comparison_df['R2_original'], width, label='Original', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['R2_optimized'], width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Configuration')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Score: Original vs Optimized')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].bar(x - width/2, comparison_df['Spearman_original'], width, label='Original', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['Spearman_optimized'], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Spearman ρ')
axes[1].set_title('Spearman ρ: Original vs Optimized')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Define dataset name
dataset_name = "Oensingen_2018-19"

# Create dataset subfolder inside existing xgboost_params folder
save_dir = os.path.join('xgboost_params', dataset_name)
os.makedirs(save_dir, exist_ok=True)

# Save comparison dataframe
comparison_df.to_csv(os.path.join(save_dir, 'optimization_comparison.csv'), index=False)

# Save both original and optimized parameters as JSON
with open(os.path.join(save_dir, 'original_hyperparameters.json'), 'w') as f:
    json.dump(all_original_params, f, indent=4)

with open(os.path.join(save_dir, 'optimized_hyperparameters.json'), 'w') as f:
    json.dump(all_best_params, f, indent=4)

# ADD THIS: Save best performing parameters (choose better of original vs optimized)
all_best_performing_params = {}
for _, row in comparison_df.iterrows():
    config_name = f"{row['Temporal']}_{row['Predictors']}"
    if row['Better_Model'] == 'Optimized':
        all_best_performing_params[config_name] = all_best_params[config_name]
    else:
        all_best_performing_params[config_name] = all_original_params[config_name]

with open(os.path.join(save_dir, 'best_performing_hyperparameters.json'), 'w') as f:
    json.dump(all_best_performing_params, f, indent=4)

# Save each configuration's params as separate JSON files
for config_name in all_best_params.keys():
    # Original params
    with open(os.path.join(save_dir, f'{config_name}_original_params.json'), 'w') as f:
        json.dump(all_original_params[config_name], f, indent=4)
    
    # Optimized params
    with open(os.path.join(save_dir, f'{config_name}_optimized_params.json'), 'w') as f:
        json.dump(all_best_params[config_name], f, indent=4)
    
    # Best performing params
    with open(os.path.join(save_dir, f'{config_name}_best_params.json'), 'w') as f:
        json.dump(all_best_performing_params[config_name], f, indent=4)

# Save detailed comparison with both parameter sets
comparison_with_params = comparison_df.copy()
comparison_with_params['Original_Params'] = comparison_with_params.apply(
    lambda row: all_original_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Optimized_Params'] = comparison_with_params.apply(
    lambda row: all_best_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Best_Performing_Params'] = comparison_with_params.apply(
    lambda row: all_best_performing_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params.to_csv(os.path.join(save_dir, 'detailed_comparison_with_params.csv'), index=False)

print(f"\nResults saved to {save_dir}!")
print("  - optimization_comparison.csv")
print("  - original_hyperparameters.json")
print("  - optimized_hyperparameters.json")
print("  - best_performing_hyperparameters.json (automatically selects better model)")
print("  - detailed_comparison_with_params.csv")
print("  - Individual config param files (18 files: 6 original + 6 optimized + 6 best)")
print(f"\nTotal files saved: {len(os.listdir(save_dir))}")
print(f"Full path: {os.path.abspath(save_dir)}")

# Print summary of which models performed better
print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)
for _, row in comparison_df.iterrows():
    winner = "✓ OPTIMIZED" if row['Better_Model'] == 'Optimized' else "✓ ORIGINAL"
    print(f"{row['Temporal']:15} | {row['Predictors']:20} | {winner:12} | ΔR²: {row['R2_improvement']:+.4f}")

# Experiment 5: Forel dataset

In [ ]:
# FIX TIMESTAMP
forel_lag['Timestamp'] = pd.to_datetime(forel_lag['Timestamp'])
forel_lag = forel_lag.set_index('Timestamp').sort_index()

forel_daily['Date'] = pd.to_datetime(forel_daily['Date'])
forel_daily = forel_daily.set_index('Date').sort_index()

In [ ]:
# Define base and lag predictors
base_predictors = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "Fertilizer_N_kg_ha"
]

lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO", 

    # --- Meteorological lag variables ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",

    # --- Management variables ---
    "Mowing", "FertilizerOrganic", "FertilizerMineral",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilWater_30cm_roll3d_mean", "SoilWater_30cm_roll5d_mean", "SoilWater_30cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "SoilTemp_30cm_roll3d_mean", "SoilTemp_30cm_roll5d_mean", "SoilTemp_30cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilWater_30cm_roll3d_sum", "SoilWater_30cm_roll5d_sum", "SoilWater_30cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "SoilTemp_30cm_roll3d_sum", "SoilTemp_30cm_roll5d_sum", "SoilTemp_30cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # Management events and days since
    "Mowing", "FertilizerOrganic", "FertilizerMineral",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral", "Fertilizer_N_kg_ha", "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d" , "Fertilizer_N_kg_ha_expHL14d"
]

In [ ]:
configurations = [
    ("Hourly", "Base", forel_lag, base_predictors),
    ("Hourly", "Lag", forel_lag, lag_predictors),
    ("Hourly", "Augmented", forel_lag, augmented_predictors),
    ("Daily", "Base", forel_daily, base_predictors),
    ("Daily", "Lag", forel_daily, lag_predictors),
    ("Daily", "Augmented", forel_daily, augmented_predictors),
]

all_optimized_results = []
all_best_params = {}

for temporal, pred_type, df, predictors in configurations:
    print("\n" + "="*80)
    print(f"OPTIMIZING: {temporal} | {pred_type}")
    print("="*80)
    
    # CORRECTED: Only 7 return values now
    best_params, search_results, X_trainval, y_trainval, X_test, y_test, offset = hyperparameter_search_timeseries(
        df=df, 
        predictors=predictors, 
        target="N2O_Flux_ln", 
        test_size=0.1,
        search_type='bayesian',
        n_iter=500,
        random_state=42,
        n_cv_splits=9
    )
    
    all_best_params[f"{temporal}_{pred_type}"] = best_params
    
    # CORRECTED: train_final_model also needs updating
    final_results = train_final_model(
        X_trainval=X_trainval,  # Changed from X_train
        y_trainval=y_trainval,  # Changed from y_train
        X_test=X_test,
        y_test=y_test,
        offset=offset,
        predictors=predictors,
        best_params=best_params,
        random_state=42,
        plot=True
    )
    
    all_optimized_results.append({
        'Temporal': temporal, 
        'Predictors': pred_type, 
        'R2_optimized': final_results['r2'], 
        'Spearman_optimized': final_results['spearman_rho'], 
        'Best_Params': best_params
    })
    
    print(f"\n✓ Completed {temporal} | {pred_type}")
    print(f"  Test R²: {final_results['r2']:.4f}")
    print(f"  Test ρ: {final_results['spearman_rho']:.4f}")

In [ ]:
# Re-run original models for comparison
original_results = []
all_original_params = {}  # ADD THIS: Store original parameters

for temporal, pred_type, df, predictors in configurations:
    print(f"\nRunning original {temporal} | {pred_type}...")
    
    orig = train_xgb_timeseries(
        df=df,
        predictors=predictors,
        target="N2O_Flux_ln",
        test_size=0.1,
        n_estimators=300,
        random_state=42,
        plot=False
    )
    
    # Store original parameters
    config_name = f"{temporal}_{pred_type}"
    all_original_params[config_name] = {
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.1,  # ← FIXED
        'subsample': 0.8,      # ← FIXED
        'colsample_bytree': 0.8,  # ← FIXED
        'min_child_weight': 1,
        'gamma': 0,
        'random_state': 42
    }
    
    original_results.append({
        'Temporal': temporal,
        'Predictors': pred_type,
        'R2_original': orig['r2'],
        'Spearman_original': orig['spearman_rho']
    })

# Combine results
comparison_df = pd.merge(
    pd.DataFrame(original_results),
    pd.DataFrame(all_optimized_results),
    on=['Temporal', 'Predictors']
)

# Calculate improvements
comparison_df['R2_improvement'] = comparison_df['R2_optimized'] - comparison_df['R2_original']
comparison_df['Spearman_improvement'] = comparison_df['Spearman_optimized'] - comparison_df['Spearman_original']

# ADD THIS: Flag which model is better
comparison_df['Better_Model'] = comparison_df.apply(
    lambda row: 'Optimized' if row['R2_optimized'] > row['R2_original'] else 'Original', 
    axis=1
)

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ORIGINAL VS OPTIMIZED")
print("="*80)
print(comparison_df[['Temporal', 'Predictors', 'R2_original', 'R2_optimized', 
                     'R2_improvement', 'Spearman_original', 'Spearman_optimized', 
                     'Spearman_improvement', 'Better_Model']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x - width/2, comparison_df['R2_original'], width, label='Original', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['R2_optimized'], width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Configuration')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Score: Original vs Optimized')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].bar(x - width/2, comparison_df['Spearman_original'], width, label='Original', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['Spearman_optimized'], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Spearman ρ')
axes[1].set_title('Spearman ρ: Original vs Optimized')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Define dataset name
dataset_name = "Forel"

# Create dataset subfolder inside existing xgboost_params folder
save_dir = os.path.join('xgboost_params', dataset_name)
os.makedirs(save_dir, exist_ok=True)

# Save comparison dataframe
comparison_df.to_csv(os.path.join(save_dir, 'optimization_comparison.csv'), index=False)

# Save both original and optimized parameters as JSON
with open(os.path.join(save_dir, 'original_hyperparameters.json'), 'w') as f:
    json.dump(all_original_params, f, indent=4)

with open(os.path.join(save_dir, 'optimized_hyperparameters.json'), 'w') as f:
    json.dump(all_best_params, f, indent=4)

# ADD THIS: Save best performing parameters (choose better of original vs optimized)
all_best_performing_params = {}
for _, row in comparison_df.iterrows():
    config_name = f"{row['Temporal']}_{row['Predictors']}"
    if row['Better_Model'] == 'Optimized':
        all_best_performing_params[config_name] = all_best_params[config_name]
    else:
        all_best_performing_params[config_name] = all_original_params[config_name]

with open(os.path.join(save_dir, 'best_performing_hyperparameters.json'), 'w') as f:
    json.dump(all_best_performing_params, f, indent=4)

# Save each configuration's params as separate JSON files
for config_name in all_best_params.keys():
    # Original params
    with open(os.path.join(save_dir, f'{config_name}_original_params.json'), 'w') as f:
        json.dump(all_original_params[config_name], f, indent=4)
    
    # Optimized params
    with open(os.path.join(save_dir, f'{config_name}_optimized_params.json'), 'w') as f:
        json.dump(all_best_params[config_name], f, indent=4)
    
    # Best performing params
    with open(os.path.join(save_dir, f'{config_name}_best_params.json'), 'w') as f:
        json.dump(all_best_performing_params[config_name], f, indent=4)

# Save detailed comparison with both parameter sets
comparison_with_params = comparison_df.copy()
comparison_with_params['Original_Params'] = comparison_with_params.apply(
    lambda row: all_original_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Optimized_Params'] = comparison_with_params.apply(
    lambda row: all_best_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Best_Performing_Params'] = comparison_with_params.apply(
    lambda row: all_best_performing_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params.to_csv(os.path.join(save_dir, 'detailed_comparison_with_params.csv'), index=False)

print(f"\nResults saved to {save_dir}!")
print("  - optimization_comparison.csv")
print("  - original_hyperparameters.json")
print("  - optimized_hyperparameters.json")
print("  - best_performing_hyperparameters.json (automatically selects better model)")
print("  - detailed_comparison_with_params.csv")
print("  - Individual config param files (18 files: 6 original + 6 optimized + 6 best)")
print(f"\nTotal files saved: {len(os.listdir(save_dir))}")
print(f"Full path: {os.path.abspath(save_dir)}")

# Print summary of which models performed better
print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)
for _, row in comparison_df.iterrows():
    winner = "✓ OPTIMIZED" if row['Better_Model'] == 'Optimized' else "✓ ORIGINAL"
    print(f"{row['Temporal']:15} | {row['Predictors']:20} | {winner:12} | ΔR²: {row['R2_improvement']:+.4f}")

# Experiment 6: Tanikon dataset

In [ ]:
# FIX TIMESTAMP
tanikon_lag['Timestamp'] = pd.to_datetime(tanikon_lag['Timestamp'])
tanikon_lag = tanikon_lag.set_index('Timestamp').sort_index()

tanikon_daily['Date'] = pd.to_datetime(tanikon_daily['Date'])
tanikon_daily = tanikon_daily.set_index('Date').sort_index()

In [ ]:
base_predictors = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
]

hourly_lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilWater_30cm_lag1d_daily", "SoilWater_30cm_lag3d_daily", "SoilWater_30cm_lag5d_daily", "SoilWater_30cm_lag7d_daily",
    "SoilTemp_5cm_lag1d_daily", "SoilTemp_5cm_lag3d_daily", "SoilTemp_5cm_lag5d_daily", "SoilTemp_5cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "SoilTemp_30cm_lag1d_daily", "SoilTemp_30cm_lag3d_daily", "SoilTemp_30cm_lag5d_daily", "SoilTemp_30cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",

    # --- Management variables ---
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation"
]

hourly_augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d_daily", "NEE_lag3d_daily", "NEE_lag5d_daily", "NEE_lag7d_daily",
    "GPP_lag1d_daily", "GPP_lag3d_daily", "GPP_lag5d_daily", "GPP_lag7d_daily",
    "RECO_lag1d_daily", "RECO_lag3d_daily", "RECO_lag5d_daily", "RECO_lag7d_daily",
    "SolarRadiation_lag1d_daily", "SolarRadiation_lag3d_daily", "SolarRadiation_lag5d_daily", "SolarRadiation_lag7d_daily",
    "AirTemp_lag1d_daily", "AirTemp_lag3d_daily", "AirTemp_lag5d_daily", "AirTemp_lag7d_daily",
    "VPD_lag1d_daily", "VPD_lag3d_daily", "VPD_lag5d_daily", "VPD_lag7d_daily",
    "SoilWater_5cm_lag1d_daily", "SoilWater_5cm_lag3d_daily", "SoilWater_5cm_lag5d_daily", "SoilWater_5cm_lag7d_daily",
    "SoilWater_15cm_lag1d_daily", "SoilWater_15cm_lag3d_daily", "SoilWater_15cm_lag5d_daily", "SoilWater_15cm_lag7d_daily",
    "SoilWater_30cm_lag1d_daily", "SoilWater_30cm_lag3d_daily", "SoilWater_30cm_lag5d_daily", "SoilWater_30cm_lag7d_daily",
    "SoilTemp_5cm_lag1d_daily", "SoilTemp_5cm_lag3d_daily", "SoilTemp_5cm_lag5d_daily", "SoilTemp_5cm_lag7d_daily",
    "SoilTemp_15cm_lag1d_daily", "SoilTemp_15cm_lag3d_daily", "SoilTemp_15cm_lag5d_daily", "SoilTemp_15cm_lag7d_daily",
    "SoilTemp_30cm_lag1d_daily", "SoilTemp_30cm_lag3d_daily", "SoilTemp_30cm_lag5d_daily", "SoilTemp_30cm_lag7d_daily",
    "Precipitation_lag1d_daily", "Precipitation_lag3d_daily", "Precipitation_lag5d_daily", "Precipitation_lag7d_daily",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilWater_30cm_roll3d_mean", "SoilWater_30cm_roll5d_mean", "SoilWater_30cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "SoilTemp_30cm_roll3d_mean", "SoilTemp_30cm_roll5d_mean", "SoilTemp_30cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilWater_30cm_roll3d_sum", "SoilWater_30cm_roll5d_sum", "SoilWater_30cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "SoilTemp_30cm_roll3d_sum", "SoilTemp_30cm_roll5d_sum", "SoilTemp_30cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # Management events and days since
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation"
]

daily_lag_predictors = [
    # --- Meteorological base variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",

    # --- Meteorological lag variables ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",

    # --- Management variables ---
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation"
]

daily_augmented_predictors = [
    # --- Base current-day variables ---
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm", "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO",
    
    # --- Lag features (1, 3, 5, 7 days) ---
    "NEE_lag1d", "NEE_lag3d", "NEE_lag5d", "NEE_lag7d",
    "GPP_lag1d", "GPP_lag3d", "GPP_lag5d", "GPP_lag7d",
    "RECO_lag1d", "RECO_lag3d", "RECO_lag5d", "RECO_lag7d",
    "SolarRadiation_lag1d", "SolarRadiation_lag3d", "SolarRadiation_lag5d", "SolarRadiation_lag7d",
    "AirTemp_lag1d", "AirTemp_lag3d", "AirTemp_lag5d", "AirTemp_lag7d",
    "VPD_lag1d", "VPD_lag3d", "VPD_lag5d", "VPD_lag7d",
    "SoilWater_5cm_lag1d", "SoilWater_5cm_lag3d", "SoilWater_5cm_lag5d", "SoilWater_5cm_lag7d",
    "SoilWater_15cm_lag1d", "SoilWater_15cm_lag3d", "SoilWater_15cm_lag5d", "SoilWater_15cm_lag7d",
    "SoilWater_30cm_lag1d", "SoilWater_30cm_lag3d", "SoilWater_30cm_lag5d", "SoilWater_30cm_lag7d",
    "SoilTemp_5cm_lag1d", "SoilTemp_5cm_lag3d", "SoilTemp_5cm_lag5d", "SoilTemp_5cm_lag7d",
    "SoilTemp_15cm_lag1d", "SoilTemp_15cm_lag3d", "SoilTemp_15cm_lag5d", "SoilTemp_15cm_lag7d",
    "SoilTemp_30cm_lag1d", "SoilTemp_30cm_lag3d", "SoilTemp_30cm_lag5d", "SoilTemp_30cm_lag7d",
    "Precipitation_lag1d", "Precipitation_lag3d", "Precipitation_lag5d", "Precipitation_lag7d",
    
    # --- Rolling mean features (3, 5, 7 day windows) ---
    "NEE_roll3d_mean", "NEE_roll5d_mean", "NEE_roll7d_mean",
    "GPP_roll3d_mean", "GPP_roll5d_mean", "GPP_roll7d_mean",
    "RECO_roll3d_mean", "RECO_roll5d_mean", "RECO_roll7d_mean",
    "SolarRadiation_roll3d_mean", "SolarRadiation_roll5d_mean", "SolarRadiation_roll7d_mean",
    "AirTemp_roll3d_mean", "AirTemp_roll5d_mean", "AirTemp_roll7d_mean",
    "VPD_roll3d_mean", "VPD_roll5d_mean", "VPD_roll7d_mean",
    "SoilWater_5cm_roll3d_mean", "SoilWater_5cm_roll5d_mean", "SoilWater_5cm_roll7d_mean",
    "SoilWater_15cm_roll3d_mean", "SoilWater_15cm_roll5d_mean", "SoilWater_15cm_roll7d_mean",
    "SoilWater_30cm_roll3d_mean", "SoilWater_30cm_roll5d_mean", "SoilWater_30cm_roll7d_mean",
    "SoilTemp_5cm_roll3d_mean", "SoilTemp_5cm_roll5d_mean", "SoilTemp_5cm_roll7d_mean",
    "SoilTemp_15cm_roll3d_mean", "SoilTemp_15cm_roll5d_mean", "SoilTemp_15cm_roll7d_mean",
    "SoilTemp_30cm_roll3d_mean", "SoilTemp_30cm_roll5d_mean", "SoilTemp_30cm_roll7d_mean",
    "Precipitation_roll3d_mean", "Precipitation_roll5d_mean", "Precipitation_roll7d_mean",
    
    # --- Rolling sum features (3, 5, 7 day windows) ---
    "NEE_roll3d_sum", "NEE_roll5d_sum", "NEE_roll7d_sum",
    "GPP_roll3d_sum", "GPP_roll5d_sum", "GPP_roll7d_sum",
    "RECO_roll3d_sum", "RECO_roll5d_sum", "RECO_roll7d_sum",
    "SolarRadiation_roll3d_sum", "SolarRadiation_roll5d_sum", "SolarRadiation_roll7d_sum",
    "AirTemp_roll3d_sum", "AirTemp_roll5d_sum", "AirTemp_roll7d_sum",
    "VPD_roll3d_sum", "VPD_roll5d_sum", "VPD_roll7d_sum",
    "SoilWater_5cm_roll3d_sum", "SoilWater_5cm_roll5d_sum", "SoilWater_5cm_roll7d_sum",
    "SoilWater_15cm_roll3d_sum", "SoilWater_15cm_roll5d_sum", "SoilWater_15cm_roll7d_sum",
    "SoilWater_30cm_roll3d_sum", "SoilWater_30cm_roll5d_sum", "SoilWater_30cm_roll7d_sum",
    "SoilTemp_5cm_roll3d_sum", "SoilTemp_5cm_roll5d_sum", "SoilTemp_5cm_roll7d_sum",
    "SoilTemp_15cm_roll3d_sum", "SoilTemp_15cm_roll5d_sum", "SoilTemp_15cm_roll7d_sum",
    "SoilTemp_30cm_roll3d_sum", "SoilTemp_30cm_roll5d_sum", "SoilTemp_30cm_roll7d_sum",
    "Precipitation_roll3d_sum", "Precipitation_roll5d_sum", "Precipitation_roll7d_sum",
    
    # Management events and days since
    "Mowing", "FertilizerOrganic", "FertilizerMineral", "SoilCultivation",
    "DaysSince_Mowing", "DaysSince_FertilizerOrganic", "DaysSince_FertilizerMineral",
    "DaysSince_SoilCultivation"
]

In [ ]:
configurations = [
    ("Hourly", "Base", tanikon_lag, base_predictors),
    ("Hourly", "Lag", tanikon_lag, hourly_lag_predictors),
    ("Hourly", "Augmented", tanikon_lag, hourly_augmented_predictors),
    ("Daily", "Base", tanikon_daily, base_predictors),
    ("Daily", "Lag", tanikon_daily, daily_lag_predictors),
    ("Daily", "Augmented", tanikon_daily, daily_augmented_predictors),
]

all_optimized_results = []
all_best_params = {}

for temporal, pred_type, df, predictors in configurations:
    print("\n" + "="*80)
    print(f"OPTIMIZING: {temporal} | {pred_type}")
    print("="*80)
    
    # CORRECTED: Only 7 return values now
    best_params, search_results, X_trainval, y_trainval, X_test, y_test, offset = hyperparameter_search_timeseries(
        df=df, 
        predictors=predictors, 
        target="N2O_Flux_ln", 
        test_size=0.1,
        search_type='bayesian',
        n_iter=500,
        random_state=42,
        n_cv_splits=9
    )
    
    all_best_params[f"{temporal}_{pred_type}"] = best_params
    
    # CORRECTED: train_final_model also needs updating
    final_results = train_final_model(
        X_trainval=X_trainval,  # Changed from X_train
        y_trainval=y_trainval,  # Changed from y_train
        X_test=X_test,
        y_test=y_test,
        offset=offset,
        predictors=predictors,
        best_params=best_params,
        random_state=42,
        plot=True
    )
    
    all_optimized_results.append({
        'Temporal': temporal, 
        'Predictors': pred_type, 
        'R2_optimized': final_results['r2'], 
        'Spearman_optimized': final_results['spearman_rho'], 
        'Best_Params': best_params
    })
    
    print(f"\n✓ Completed {temporal} | {pred_type}")
    print(f"  Test R²: {final_results['r2']:.4f}")
    print(f"  Test ρ: {final_results['spearman_rho']:.4f}")

In [ ]:
# Re-run original models for comparison
original_results = []
all_original_params = {}  # ADD THIS: Store original parameters

for temporal, pred_type, df, predictors in configurations:
    print(f"\nRunning original {temporal} | {pred_type}...")
    
    orig = train_xgb_timeseries(
        df=df,
        predictors=predictors,
        target="N2O_Flux_ln",
        test_size=0.1,
        n_estimators=300,
        random_state=42,
        plot=False
    )
    
    # Store original parameters
    config_name = f"{temporal}_{pred_type}"
    all_original_params[config_name] = {
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.1,  # ← FIXED
        'subsample': 0.8,      # ← FIXED
        'colsample_bytree': 0.8,  # ← FIXED
        'min_child_weight': 1,
        'gamma': 0,
        'random_state': 42
    }
    
    original_results.append({
        'Temporal': temporal,
        'Predictors': pred_type,
        'R2_original': orig['r2'],
        'Spearman_original': orig['spearman_rho']
    })

# Combine results
comparison_df = pd.merge(
    pd.DataFrame(original_results),
    pd.DataFrame(all_optimized_results),
    on=['Temporal', 'Predictors']
)

# Calculate improvements
comparison_df['R2_improvement'] = comparison_df['R2_optimized'] - comparison_df['R2_original']
comparison_df['Spearman_improvement'] = comparison_df['Spearman_optimized'] - comparison_df['Spearman_original']

# ADD THIS: Flag which model is better
comparison_df['Better_Model'] = comparison_df.apply(
    lambda row: 'Optimized' if row['R2_optimized'] > row['R2_original'] else 'Original', 
    axis=1
)

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ORIGINAL VS OPTIMIZED")
print("="*80)
print(comparison_df[['Temporal', 'Predictors', 'R2_original', 'R2_optimized', 
                     'R2_improvement', 'Spearman_original', 'Spearman_optimized', 
                     'Spearman_improvement', 'Better_Model']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x - width/2, comparison_df['R2_original'], width, label='Original', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['R2_optimized'], width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Configuration')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Score: Original vs Optimized')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].bar(x - width/2, comparison_df['Spearman_original'], width, label='Original', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['Spearman_optimized'], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Spearman ρ')
axes[1].set_title('Spearman ρ: Original vs Optimized')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{row['Temporal']}\n{row['Predictors']}" for _, row in comparison_df.iterrows()], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Define dataset name
dataset_name = "Tanikon"

# Create dataset subfolder inside existing xgboost_params folder
save_dir = os.path.join('xgboost_params', dataset_name)
os.makedirs(save_dir, exist_ok=True)

# Save comparison dataframe
comparison_df.to_csv(os.path.join(save_dir, 'optimization_comparison.csv'), index=False)

# Save both original and optimized parameters as JSON
with open(os.path.join(save_dir, 'original_hyperparameters.json'), 'w') as f:
    json.dump(all_original_params, f, indent=4)

with open(os.path.join(save_dir, 'optimized_hyperparameters.json'), 'w') as f:
    json.dump(all_best_params, f, indent=4)

# ADD THIS: Save best performing parameters (choose better of original vs optimized)
all_best_performing_params = {}
for _, row in comparison_df.iterrows():
    config_name = f"{row['Temporal']}_{row['Predictors']}"
    if row['Better_Model'] == 'Optimized':
        all_best_performing_params[config_name] = all_best_params[config_name]
    else:
        all_best_performing_params[config_name] = all_original_params[config_name]

with open(os.path.join(save_dir, 'best_performing_hyperparameters.json'), 'w') as f:
    json.dump(all_best_performing_params, f, indent=4)

# Save each configuration's params as separate JSON files
for config_name in all_best_params.keys():
    # Original params
    with open(os.path.join(save_dir, f'{config_name}_original_params.json'), 'w') as f:
        json.dump(all_original_params[config_name], f, indent=4)
    
    # Optimized params
    with open(os.path.join(save_dir, f'{config_name}_optimized_params.json'), 'w') as f:
        json.dump(all_best_params[config_name], f, indent=4)
    
    # Best performing params
    with open(os.path.join(save_dir, f'{config_name}_best_params.json'), 'w') as f:
        json.dump(all_best_performing_params[config_name], f, indent=4)

# Save detailed comparison with both parameter sets
comparison_with_params = comparison_df.copy()
comparison_with_params['Original_Params'] = comparison_with_params.apply(
    lambda row: all_original_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Optimized_Params'] = comparison_with_params.apply(
    lambda row: all_best_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params['Best_Performing_Params'] = comparison_with_params.apply(
    lambda row: all_best_performing_params[f"{row['Temporal']}_{row['Predictors']}"], axis=1
)
comparison_with_params.to_csv(os.path.join(save_dir, 'detailed_comparison_with_params.csv'), index=False)

print(f"\nResults saved to {save_dir}!")
print("  - optimization_comparison.csv")
print("  - original_hyperparameters.json")
print("  - optimized_hyperparameters.json")
print("  - best_performing_hyperparameters.json (automatically selects better model)")
print("  - detailed_comparison_with_params.csv")
print("  - Individual config param files (18 files: 6 original + 6 optimized + 6 best)")
print(f"\nTotal files saved: {len(os.listdir(save_dir))}")
print(f"Full path: {os.path.abspath(save_dir)}")

# Print summary of which models performed better
print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)
for _, row in comparison_df.iterrows():
    winner = "✓ OPTIMIZED" if row['Better_Model'] == 'Optimized' else "✓ ORIGINAL"
    print(f"{row['Temporal']:15} | {row['Predictors']:20} | {winner:12} | ΔR²: {row['R2_improvement']:+.4f}")